In [6]:
import pandas as pd
from main import run_ga
from tqdm import trange
from sympy import parse_expr
import sympy as sp
import numpy as np
from pathlib import Path

SEED = 56
df = pd.read_csv("dataset/FeynmanEquations.csv")
df.dropna(axis=0, how="all", inplace=True)
df.Formula = df.Formula.apply(lambda s: s.replace("arcsin", "asin").replace("arccos", "acos").replace("arctan", "atan"))
# dataset fixes/hacks present in original code
df.loc[18, "Filename"] = "I.15.10"
df.loc[82, "# variables"] = 3
df.loc[49, "Filename"] = "I.48.20"
df.loc[61, "Filename"] = "II.11.7"
df.head()


,Filename,Number,Output,Formula,# variables,v1_name,v1_low,v1_high,v2_name,v2_low,...,v7_high,v8_name,v8_low,v8_high,v9_name,v9_low,v9_high,v10_name,v10_low,v10_high
0,I.6.2a,1.0,f,exp(-theta**2/2)/sqrt(2*pi),1.0,theta,1.0,3.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,I.6.2,2.0,f,exp(-(theta/sigma)**2/2)/(sqrt(2*pi)*sigma),2.0,sigma,1.0,3.0,theta,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,I.6.2b,3.0,f,exp(-((theta-theta1)/sigma)**2/2)/(sqrt(2*pi)*...,3.0,sigma,1.0,3.0,theta,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,I.8.14,4.0,d,sqrt((x2-x1)**2+(y2-y1)**2),4.0,x1,1.0,5.0,x2,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,I.9.18,5.0,F,G*m1*m2/((x2-x1)**2+(y2-y1)**2+(z2-z1)**2),9.0,m1,1.0,2.0,m2,1.0,...,2.0,z1,3.0,4.0,z2,1.0,2.0,NaN,NaN,NaN


In [ ]:
n = len(df)
results = []
verbose = False  # Set to True to enable per-equation printing

for i in trange(1):
    vars_list = []
    for j in range(int(df["# variables"][i])):
        var_name = df[f"v{j+1}_name"][i]
        vars_list.append(sp.Symbol(var_name, real=True))
    
    expr = parse_expr(
        df.Formula[i], 
        local_dict={str(var): var for var in vars_list}, 
        evaluate=False
    )
    dataset = pd.read_csv(
        f"dataset/Feynman_with_units/{df['Filename'][i]}",
        header=None,
        sep=r"\s+",
    )
    dataset = dataset.sample(frac=1).reset_index(drop=True)
    dataset.dropna(axis=1, how="all", inplace=True)
    y = dataset.iloc[:, -1].to_numpy()
    var_names = [str(var) for var in vars_list]
    X = {
        name: dataset.iloc[:, idx].to_numpy(dtype=float)
        for idx, name in enumerate(var_names)
    }
    unary_funcs = {
        'sin': (np.sin, sp.sin),
        'cos': (np.cos, sp.cos),
        'tan': (np.tan, sp.tan),
        'exp': (np.exp, sp.exp),
        'sqrt': (np.sqrt, sp.sqrt)
    }
    best, best_r2 = run_ga(
        X,
        y,
        pop_size=500,
        gens=30,
        max_init_depth=4,
        max_depth=10,
        crossover_rate=0.85,
        mutation_rate=0.2,
        tournament_k=5,
        elitism=15,
        var_names=var_names,
        unary_funcs=unary_funcs,
        use_learned_crossover=True,
        selector_lr=0.0005,
        mlp_layers=6,
        mlp_hidden_dim=128,
        mlp_activation="gelu",
        mlp_dropout=0.1,
        stopping_metric="r2",
        stopping_threshold=1.0,
        complexity_coef=1e-4,
        inclusion_coef=0.5,
        prob_const=0.35,
        const_range=(-3, 3),
        point_mutation_prob=0.1,
        selector_update_freq=3,
        temperature=0.7,
        critic_weight=0.6,
        entropy_weight=0.02,
        loss_metric="mse",
        use_pareto=False,
        opt_constants=True,
        const_opt_freq=5,
        verbose=True,
        print_freq=1,
        seed=SEED,
    )
    
    found_formula = None if best is None else best.to_string()
    # Store results
    results.append({
        'equation_id': i + 1,
        'filename': df['Filename'][i],
        'target_formula': df.Formula[i],
        'found_formula': found_formula,
        'r2_score': best_r2,
        'num_variables': int(df["# variables"][i])
    })
    
    if verbose:
        print(f"\n{'='*60}")
        print(f"Equation {i+1}/{n}: {df['Filename'][i]}")
        print(f"Target: {df.Formula[i]}")
        print(f"Found: {found_formula}")
        if best_r2 is None:
            print("R² Score: None")
        else:
            print(f"R² Score: {best_r2:.6f}")
        print(f"{'='*60}\n")

# Calculate and display aggregate metrics
results_df = pd.DataFrame(results)
print("\n" + "="*80)
print("FINAL DATASET PERFORMANCE METRICS")
print("="*80)
print(f"\nTotal Equations Tested: {len(results_df)}")
print(f"Mean R² Score: {results_df['r2_score'].mean():.6f}")
print(f"Median R² Score: {results_df['r2_score'].median():.6f}")
print(f"Std Dev R² Score: {results_df['r2_score'].std():.6f}")
print(f"Min R² Score: {results_df['r2_score'].min():.6f}")
print(f"Max R² Score: {results_df['r2_score'].max():.6f}")

# Success rate metrics
excellent = (results_df['r2_score'] >= 0.99).sum()
good = ((results_df['r2_score'] >= 0.95) & (results_df['r2_score'] < 0.99)).sum()
moderate = ((results_df['r2_score'] >= 0.90) & (results_df['r2_score'] < 0.95)).sum()
poor = (results_df['r2_score'] < 0.90).sum()

print(f"\nSuccess Rate Breakdown:")
print(f"  Excellent (R² ≥ 0.99): {excellent}/{n} ({100*excellent/n:.1f}%)")
print(f"  Good (0.95 ≤ R² < 0.99): {good}/{n} ({100*good/n:.1f}%)")
print(f"  Moderate (0.90 ≤ R² < 0.95): {moderate}/{n} ({100*moderate/n:.1f}%)")
print(f"  Poor (R² < 0.90): {poor}/{n} ({100*poor/n:.1f}%)")

  0%|          | 0/1 [00:00<?, ?it/s]


TypeError: run_ga() got an unexpected keyword argument 'loss_metric'

: 